# Backpropagation — Cleaned Notebook

This notebook is a cleaned and improved version of `01_Backpropagation.ipynb`. Goals: reproducibility (fixed seeds), clearer pedagogy (markdown headings + short explanations), modern Keras API (explicit Input layers), use of validation and callbacks, plotted learning curves, and a corrected perceptron demonstration.

Original: https://github.com/aditya01-dev/Deep-Learning/blob/main/01_Backpropagation.ipynb

In [ ]:
# Basic imports and reproducibility
import numpy as np
import random
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print('numpy:', np.__version__)
print('tensorflow:', tf.__version__)


In [ ]:
# Helper to plot training history
def plot_history(history, title=None):
    if hasattr(history, 'history'):
        h = history.history
    else:
        h = history
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(h.get('loss', []), label='loss')
    if 'val_loss' in h: plt.plot(h['val_loss'], label='val_loss')
    plt.legend(); plt.title('Loss')

    plt.subplot(1,2,2)
    if 'accuracy' in h or 'acc' in h:
        acc = h.get('accuracy', h.get('acc'))
        plt.plot(acc, label='accuracy')
        if 'val_accuracy' in h or 'val_acc' in h:
            plt.plot(h.get('val_accuracy', h.get('val_acc')), label='val_accuracy')
        plt.legend(); plt.title('Accuracy')
    if title: plt.suptitle(title)
    plt.show()


## Example 1 — MNIST (simple dense network)
We train a small dense network on MNIST. We use explicit Input, float32 normalization, a validation split and EarlyStopping to demonstrate best practices.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist

# Load and preprocess
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Model
model = Sequential([
    Input(shape=(28,28)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
history = model.fit(x_train, y_train, validation_split=0.1, epochs=20, batch_size=128, callbacks=callbacks, verbose=2)

plot_history(history, title='MNIST Dense (small)')

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

# Show a prediction example
import numpy as np
i = 0
pred = model.predict(x_test[i:i+1])
print('Predicted class:', np.argmax(pred[0]))
print('Actual class:', y_test[i])


## Example 2 — XOR with a small dense net
XOR is not linearly separable; a small network with a hidden layer can learn it. We train with a fixed seed and show rounded outputs.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
Y = np.array([[0],[1],[1],[0]], dtype=np.float32)

model_xor = Sequential([
    Input(shape=(2,)),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_xor.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_xor = model_xor.fit(X, Y, epochs=200, verbose=0)
plot_history(history_xor, title='XOR training')
loss, acc = model_xor.evaluate(X, Y, verbose=0)
print('XOR final loss, acc:', loss, acc)
preds = model_xor.predict(X)
print('Raw predictions:
', preds)
print('Rounded predictions:
', np.round(preds))


## Example 3 — Classic Perceptron (AND gate)
This cell demonstrates the perceptron learning rule: update weights only on misclassification using a threshold activation. Perceptron can learn linearly separable functions (e.g., AND).

In [ ]:
# Perceptron for AND dataset
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
Y = np.array([0,0,0,1], dtype=int)  # AND

w = np.zeros(3)  # bias + two weights
lr = 0.1

def perceptron_predict(x, w):
    x_with_bias = np.concatenate(([1.0], x))
    activation = np.dot(w, x_with_bias)
    return 1 if activation >= 0 else 0

# Training
for epoch in range(20):
    errors = 0
    for xi, yi in zip(X, Y):
        x_with_bias = np.concatenate(([1.0], xi))
        y_pred = perceptron_predict(xi, w)
        if y_pred != yi:
            update = lr * (yi - y_pred) * x_with_bias
            w += update
            errors += 1
    # stop early if perfect classification
    if errors == 0:
        print(f'Converged at epoch {epoch}')
        break

print('Final weights (bias, w1, w2):', w)
print('Predictions:')
for xi in X:
    print(xi, '->', perceptron_predict(xi, w))


## Example 4 — Linear regression with gradient descent (vectorized)
A simple demonstration of gradient descent for linear regression with a learning curve plotted.

In [ ]:
# toy linear regression y = 2x with noise
x = np.array([1,2,3,4], dtype=float)
y = 2 * x + 0.1 * np.random.randn(len(x))

w = 0.0
b = 0.0
lr = 0.01
epochs = 200
losses = []
n = len(x)

for epoch in range(epochs):
    y_pred = w * x + b
    error = y_pred - y
    loss = np.mean(error ** 2)
    losses.append(loss)
    dw = (2/n) * np.dot(error, x)
    db = (2/n) * np.sum(error)
    w -= lr * dw
    b -= lr * db

print('Learned w, b:', w, b)
plt.plot(losses)
plt.title('Linear regression training loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.show()


## Example 5 — Iris classification (multiclass)
Use a small dense network with standardized inputs and a validation split; show classification report and confusion matrix.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

iris = load_iris()
X = iris.data.astype('float32')
y = iris.target
y_cat = to_categorical(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model_iris = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(12, activation='relu'),
    Dense(3, activation='softmax')
])
model_iris.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_iris = model_iris.fit(X_train, y_train, validation_split=0.1, epochs=100, batch_size=8, callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)], verbose=0)
plot_history(history_iris, title='Iris training')
loss, acc = model_iris.evaluate(X_test, y_test, verbose=0)
print('Iris test loss, acc:', loss, acc)

# Predictions and reports
y_pred_proba = model_iris.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)
print('
Classification report:
')
print(classification_report(y_true, y_pred, target_names=iris.target_names))
print('Confusion matrix:
', confusion_matrix(y_true, y_pred))


---
Notes and further suggestions:
- For teaching, add short explanations before/after each cell.
- Encourage students to change seeds, architectures and observe effects.
- For larger models prefer ImageDataGenerator / tf.data pipelines.
